# Domain 4 · Prompt Engineering & Structured Output (20%)

The last domain, and the most *hands-on-keyboard* of the five: everything here is a
property of the **prompt** and the **schema** you send — not of the agent architecture
(D1), the tools (D2), the CLI config (D3) or the context strategy (D5).

**Two running examples, both already in your exercises:**

| Thread | Sections | Anchor exercise |
|---|---|---|
| **A code reviewer** that must be *precise* (few false positives) and *multi-pass* | 4.1, 4.2, 4.6 | `exercises/05-cicd-review/` |
| **An invoice extractor** that must be *schema-safe* and *self-correcting* | 4.3, 4.4 | `exercises/03-extraction-pipeline/` |
| Which API each of them should run on | 4.5 | both |

**How to use this notebook**
1. Run the setup cell once, then go **top to bottom** — later sections reuse earlier fixtures.
2. Every section = *verbatim guide text* → *plain-English table* → **a real Claude call you run** →
   *the wrong answers as code* → *the line in your own code* → *self-check*.
3. Every claim about what Claude does is produced **by a real call**, never by a Python `if`.
   Where an outcome is model-dependent (Haiku, tiny toy inputs), the cell says so — the
   **mechanism** is the lesson, the exact finding count is not.
4. Cost: ~20 Haiku calls with small `max_tokens`, plus **one real Message Batch** in 4.5. Pennies.

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import os, json, time

# Portable .env discovery — nearest .env walking up from the working dir (repo
# root or ccaf-prep/), then the sibling course .env. No machine-specific paths.
_candidates = [Path.cwd().parents[1] / "claude-with-anthropic-api" / ".env"]
_found = find_dotenv(filename=".env", usecwd=True)
_envfile = Path(_found) if _found else next((p for p in _candidates if p.exists()), None)
if _envfile: load_dotenv(_envfile); print(f"Loaded .env from: {_envfile}")
else: print("WARNING: no .env found — copy .env.example to .env at the repo root")
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — see .env.example"

client = Anthropic()
MODEL = os.environ.get("CLAUDE_MODEL") or "claude-haiku-4-5"   # cheap default (see convention below)
print("Using model:", MODEL)

---
## 4.1 · Design prompts with explicit criteria to improve precision and reduce false positives

> **Task Statement 4.1: Design prompts with explicit criteria to improve precision and reduce false positives**
>
> _See the official Claude Certified Architect – Foundations exam guide for this task
> statement's full description. The plain-English unpacking below is original._


**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Explicit criteria over vague instructions | Don't say *"check that comments are accurate"*. Say *"flag a comment only when the behavior it claims contradicts what the code does"*. The first is a topic; the second is a **decision rule** you could hand to a new hire. |
| "Be conservative" fails | *Conservative* is not a criterion — it gives the model **nothing new to test against**, so it just re-guesses with a hedge. Precision comes from naming the **categories** to report and the ones to skip, not from asking for less confidence. |
| "Only report high-confidence findings" fails | Same trap, plus a second one: LLM self-reported confidence is **poorly calibrated**. Filtering on it drops real bugs and keeps confident nonsense. (Same reason sample Q3 rejects the confidence-score option.) |
| False positives erode trust **across** categories | If the *style* category cries wolf 10× a day, developers stop reading the report — including the **security** findings that were right. One noisy category poisons the accurate ones. |
| Report **vs** skip lists | The prompt names both: report `security / correctness / resource-leak`; skip `naming, formatting, docstrings, local conventions`. "Skip" is doing as much work as "report". |
| Temporarily disable a high-FP category | The fix for a noisy category is not to soften the wording — it's to **turn it off in production** (stop shipping its findings) while you rework its prompt offline. Trust recovers immediately; the category comes back when it's good. |
| Severity criteria with **concrete code examples** | `high` / `medium` / `low` mean nothing until each level carries an example (*"high = a bug that corrupts or loses data, e.g. a loop that drops the last element"*). Examples are what make severity land the same way twice. |

**Run it — same file, same schema, two system prompts (2 real calls).** The *only* variable is
the criteria text. `cart.py` below has **two genuine defects** (an off-by-one that silently drops
the last item; a file handle that is never closed) and plenty of **bait**: a single-letter
variable, no docstring, a magic number, `%`-based math. Watch what each prompt comes back with.

Claude decides what to report — we only count and print what it returned.

In [ ]:
# 4.1 — EXPLICIT CRITERIA vs VAGUE HEDGING. Two REAL calls, same file, same output schema.
# The findings are Claude's; the only thing we change is the system prompt's criteria.

REVIEW_FILE = """# cart.py
1  def apply_discount(items, pct):
2      log = open("/tmp/cart.log", "a")          # opened, never closed
3      log.write("discounting\\n")
4      total = 0
5      for i in range(len(items) - 1):           # loops to len-1
6          total += items[i]["price"]
7      return total * (1 - pct / 100)
"""

# Shared structured-output tool (this IS 4.3's mechanism — we use it here so the two runs are
# comparable field-for-field instead of two blobs of prose).
def findings_tool(categories):
    return {
        "name": "report_findings",
        "description": "Emit code-review findings as structured data for the CI pipeline.",
        "input_schema": {"type": "object", "properties": {"findings": {"type": "array", "items": {
            "type": "object", "properties": {
                "line": {"type": "integer"},
                "category": ({"type": "string", "enum": categories} if categories
                             else {"type": "string"}),          # vague run: model invents categories
                "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                "issue": {"type": "string"},
                "suggested_fix": {"type": "string"}},
            "required": ["line", "category", "severity", "issue", "suggested_fix"]}}},
            "required": ["findings"]},
    }

# ---- (A) the VAGUE prompt: hedging words, no decision rule --------------------------------
VAGUE = ("You are a code reviewer. Review the file and report the problems you find. "
         "Be conservative and only report high-confidence findings.")

# ---- (B) the EXPLICIT prompt: report-list, skip-list, and a severity rubric with EXAMPLES --
CRITERIA = {                                                     # report ONLY these
    "security":      "Untrusted input reaching a dangerous sink; injection; secrets in code.",
    "correctness":   "Logic that yields a wrong result (off-by-one, wrong operator, bad branch).",
    "resource-leak": "A file/socket/handle opened and never closed.",
}
SKIP = "naming, formatting, docstrings, type hints, magic numbers, and local style conventions"
SEVERITY_RUBRIC = (
    "high   = wrong or lost data, or an exploitable hole. e.g. `for i in range(len(x) - 1)` "
    "         when every element must be processed (the last element is silently dropped).\n"
    "medium = a resource or correctness problem that degrades the system over time. e.g. "
    "         `f = open(p)` with no close() in a function called per request.\n"
    "low    = a real defect with no user-visible impact today."
)
EXPLICIT = (
    "You are a code reviewer. Report a finding ONLY if it falls in one of these categories:\n"
    + "\n".join(f"  - {k}: {v}" for k, v in CRITERIA.items())
    + f"\n\nDo NOT report: {SKIP}. A finding outside the categories above is a FALSE POSITIVE "
      "and erodes trust in the ones that are right.\n\nSeverity criteria:\n" + SEVERITY_RUBRIC
)

def review(system, categories, label):
    resp = client.messages.create(
        model=MODEL, max_tokens=700, system=system,
        tools=[findings_tool(categories)],
        tool_choice={"type": "tool", "name": "report_findings"},   # forced -> 4.3
        messages=[{"role": "user", "content": "Review this file:\n\n" + REVIEW_FILE}],
    )
    block = next(b for b in resp.content if b.type == "tool_use")
    fs = block.input.get("findings", [])
    print(f"\n--- {label}: {len(fs)} finding(s) ---")
    for f in fs:
        print(f"   L{f['line']:<3} {f['severity']:>6}  {f['category']:<14} {f['issue'][:64]}")
    return fs

vague_findings    = review(VAGUE,    None,             "VAGUE  ('be conservative, high-confidence only')")
explicit_findings = review(EXPLICIT, list(CRITERIA),   "EXPLICIT (report-list + skip-list + rubric)")

# Compare the two runs on the things 4.1 actually claims — VOCABULARY and SEVERITY.
# (Careful: do NOT just set-difference the category strings and call the remainder "false
#  positives". A capable model on a 7-line toy usually finds the two real bugs in BOTH runs — it
#  just LABELS them differently. What degrades under a vague prompt is the discipline, below.)
def sev_counts(fs):
    return {s: sum(1 for f in fs if f["severity"] == s) for s in ("high", "medium", "low")}

print(f"\nVOCABULARY")
print(f"   VAGUE    categories: {sorted({f['category'] for f in vague_findings})}   <- free text,")
print( "            re-invented on every run. You cannot aggregate these, cannot route them, and")
print( "            cannot 'disable the noisy category' (4.1) — there is no stable category to")
print( "            disable, and no dismissal histogram to group by (4.4's detected_pattern).")
print(f"   EXPLICIT categories: {sorted({f['category'] for f in explicit_findings})}   <- a closed")
print(f"            enum drawn from {sorted(CRITERIA)}. An off-topic finding is un-representable.")
print(f"\nSEVERITY")
print(f"   VAGUE   : {sev_counts(vague_findings)}   <- no rubric, so severity sorts nothing")
print(f"   EXPLICIT: {sev_counts(explicit_findings)}   <- the rubric's concrete examples put the")
print( "             data-loss bug above the leak, the same way on every run")
print(f"\nPRECISION: read the VAGUE findings above. Anything outside {sorted(CRITERIA)} ships to")
print( "   developers as noise — a style nit, a naming quibble, an invented issue. Nothing in the")
print( "   vague prompt CAN stop it: there is no report-list to be outside of. The EXPLICIT run's")
print( "   skip-list plus the enum make an off-topic finding literally un-representable.")
print("\nNOTE (honest): the MECHANISM is real and deterministic — criteria change the vocabulary,")
print("the severity discipline, and what is even representable. Whether the vague run ALSO happens")
print("to find the two real bugs on a 7-line toy is model-dependent; at 14 files it does not.")
print("See exercises/05-cicd-review/README.md, Pitfall 3.")

**Restoring trust: turn the noisy category *off*, don't soften it.** The guide's second skill is
easy to miss and is a favorite distractor. If `style` is generating 10 false positives a day, the
move is **not** "add *be careful* to the style prompt". It's:

In [ ]:
# The skill, as a two-line diff on the criteria registry (no API call — this is config, and it is
# exactly how exercises/05-cicd-review/review.py:90-96 is shaped: a dict you can delete a key from).
ALL_CRITERIA     = {**CRITERIA, "style": "Naming, formatting, docstring completeness."}
ENABLED_IN_PROD  = {k: v for k, v in ALL_CRITERIA.items() if k != "style"}   # <- style DISABLED

print("shipping to developers :", sorted(ENABLED_IN_PROD))
print("parked until its prompt is fixed:", sorted(set(ALL_CRITERIA) - set(ENABLED_IN_PROD)))
print("\nWhy: one high-FP category makes developers stop reading the WHOLE report — including the")
print("security findings that were correct. Disabling it restores trust TODAY; you re-enable it")
print("once its criteria are specific enough to earn a place back.")

**The anti-patterns (exam distractors)** — read them and feel why each is wrong:

In [ ]:
# ANTI-PATTERN 1: hedging words instead of a decision rule.
#   SYSTEM = "Review the code. Be conservative. Only report high-confidence findings."
#   -> "conservative" gives the model NOTHING to test a finding against; precision doesn't move.

# ANTI-PATTERN 2 (sample Q3 option B): filter on a self-reported confidence score.
#   findings = [f for f in findings if f["confidence"] >= 0.8]
#   -> LLM confidence is POORLY CALIBRATED. The model is already confidently wrong on the hard
#      cases, so this drops real bugs and keeps assured nonsense. Filter by CRITERIA, not by vibes.

# ANTI-PATTERN 3: keep the noisy category live while you "improve it later".
#   CRITERIA["style"] = "flag anything stylistically off"   # 10 FPs/day
#   -> developers mute the bot; your accurate security findings die with it.

# ANTI-PATTERN 4: severity levels with no examples.
#   "Use high/medium/low severity."   # every run classifies differently -> nobody trusts the sort

print("Correct: name the categories to REPORT and the ones to SKIP, and give every severity level "
      "a concrete code example. Disable a high-false-positive category in production while you fix "
      "its prompt — vague hedging and confidence filters do not buy precision.")

**In your own code — you already wrote this.** Exercise 5:

- `ccaf-prep/exercises/05-cicd-review/review.py:88-96` — the `CRITERIA` dict (`security` /
  `correctness` / `resource-leak`) with a one-line decision rule each. **This is 4.1 in three lines.**
- `ccaf-prep/exercises/05-cicd-review/review.py:125-129` — `CRITERIA_BLOCK`: the report-list *and*
  the explicit **skip** instruction (*"do not flag style/naming — vague nitpicks erode trust"*).
- `ccaf-prep/exercises/05-cicd-review/README.md:94-97` — Pitfall 3, the vague-criteria experiment.
- `ccaf-prep/exercises/01-support-agent/agent.py:36-42` — the same principle outside code review:
  the `SYSTEM` prompt states **explicit escalation criteria** (*"refunds over $500 must be
  escalated"*), which is precisely what **sample Q3** rewards over sentiment or confidence scores.

Open `review.py` and map each `CRITERIA` key to a "report vs skip" bullet in the guide.

**Self-check** (cover the answers)

1. Your reviewer reports 40% false positives. Why won't *"only report high-confidence findings"* fix it?
2. Security findings are accurate, style findings are noise. Developers now ignore **both**. What do you do first?
3. Why is `"check that comments are accurate"` a bad criterion, and what's the fixed version?
4. What makes a severity level ("high") classify the same way twice?

<details><summary>answers</summary>

1. Because "high-confidence" is not a **criterion** — it adds no rule the model can test a finding against, and LLM self-reported confidence is **poorly calibrated** anyway. Precision comes from naming the categories to **report** and to **skip**.
2. **Disable the style category in production immediately** (stop shipping its findings) and fix its prompt offline. False positives in one category erode trust in **all** of them — you cannot fix that by improving the wording while it keeps crying wolf.
3. It names a *topic*, not a *decision rule* — every reviewer draws the line somewhere different. Fixed: *"flag a comment only when the behavior it claims contradicts the actual behavior of the code."*
4. A **concrete code example attached to each level** (*"high = a loop that silently drops the last element"*). Adjectives alone drift between runs.

</details>

---
## 4.2 · Apply few-shot prompting to improve output consistency and quality

> **Task Statement 4.2: Apply few-shot prompting to improve output consistency and quality**
>
> _See the official Claude Certified Architect – Foundations exam guide for this task
> statement's full description. The plain-English unpacking below is original._


**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Few-shot beats detailed instructions | You can *describe* the output format for a paragraph and still get three different shapes in three runs. **Two examples** pin it down — the model copies a demonstrated pattern far more reliably than it follows a described one. |
| Ambiguous-case handling | The value isn't in the easy cases; it's in showing the **judgment call**: *this* SQL is fine (parameterized), *that* one is a bug (concatenated). Show the borderline case, with the reasoning. |
| Generalize, don't pattern-match | Examples teach the *principle*, not a lookup table. Show `f"...{user}"` concatenation as a bug and the model also catches `.format()` and `%` — patterns you never showed it. That's the whole reason 2–4 examples suffice. |
| 2–4 targeted examples, **with reasoning** | Not 20. Pick the ambiguous ones, and say *why* this action beat the plausible alternative — the reasoning is what transfers. |
| Format: location, issue, severity, suggested fix | The classic "actionable finding" tuple. Demonstrate it once and every subsequent finding has all four fields. |
| Acceptable pattern vs genuine issue | The **false-positive killer**: one example of a pattern that looks scary but is *fine here* teaches the model your codebase's local conventions better than a paragraph about them. |
| Extraction: informal measurements, varied structures, empty/null fields | The extraction dialect of the same idea — show `"ninety bucks"` → `90`, and a document with **no** account id → `null` (not a fabricated one). That is exactly `exercises/03-extraction-pipeline/extract.py:111-133`. |

**Run it — instructions alone vs 2 examples, twice each (4 real calls).** The file below contains
a **parameterized query (fine — the acceptable pattern)**, a **concatenated f-string query (a real
injection bug)**, and a **`.format()` query — a variant we never demonstrate**, there to test
whether the model *generalizes* or merely *pattern-matches*.

Run A gets a careful **prose description** of the format and the rule. Run B gets the same system
prompt **plus two examples**: one showing the acceptable pattern being *left alone* (with the
reasoning), one showing the genuine bug flagged in the exact format. We run each **twice** —
because the thing 4.2 is about is **consistency across runs**, which a single call can't show.

In [ ]:
# 4.2 — FEW-SHOT vs INSTRUCTIONS-ALONE. Plain TEXT output on purpose: format drift is the
# observable, and a forced schema would hide it (the schema would do few-shot's job for it).

SQL_FILE = """# users.py
1  def find_by_email(conn, email):
2      return conn.execute("SELECT * FROM users WHERE email = ?", (email,))
3
4  def find_by_name(conn, name):
5      return conn.execute(f"SELECT * FROM users WHERE name = '{name}'")
6
7  def find_by_role(conn, role):
8      return conn.execute("SELECT * FROM users WHERE role = '{}'".format(role))
"""

TASK = "Review this file and report SQL-injection findings.\n\n" + SQL_FILE

# The SAME system prompt for both runs: a careful prose spec of the rule AND the format.
SYSTEM_42 = (
    "You are a security reviewer. Report only SQL-injection issues: a query is a finding when "
    "untrusted input is interpolated into the SQL string; a query that binds parameters is safe "
    "and must not be reported. Output one line per finding in the format "
    "'L<line> | <issue> | <severity> | <suggested fix>'. If there are no findings, output 'none'."
)

# Run B adds this: 2 targeted examples. #1 is the AMBIGUOUS/ACCEPTABLE case (looks like SQL, is
# safe -> NOT flagged, with the reasoning). #2 demonstrates the exact output FORMAT on a genuine
# bug. Neither example shows .format() — line 7 tests GENERALIZATION, not recall.
FEW_SHOT = [
    {"role": "user", "content": "Review:\n1  db.execute(\"SELECT * FROM t WHERE id = ?\", (tid,))"},
    {"role": "assistant", "content":
        "none\n(The value is bound as a parameter, not concatenated into the SQL string — this is "
        "the acceptable pattern, so it is not a finding.)"},
    {"role": "user", "content": "Review:\n1  db.execute(\"SELECT * FROM t WHERE id = '\" + tid + \"'\")"},
    {"role": "assistant", "content":
        "L1 | user-controlled `tid` concatenated into the SQL string (injection) | high | "
        "bind it as a parameter: db.execute(\"SELECT * FROM t WHERE id = ?\", (tid,))"},
]

def run_42(messages_prefix, label, n=2):
    print(f"\n=== {label} ===")
    for i in range(n):
        resp = client.messages.create(
            model=MODEL, max_tokens=400, system=SYSTEM_42,
            messages=messages_prefix + [{"role": "user", "content": TASK}],
        )
        text = "".join(b.text for b in resp.content if b.type == "text").strip()
        print(f"--- run {i + 1} ---")
        for line in text.splitlines():
            if line.strip():
                print("   " + line)

run_42([],       "A · INSTRUCTIONS ALONE (prose spec of rule + format)")
run_42(FEW_SHOT, "B · SAME PROMPT + 2 FEW-SHOT EXAMPLES")

print("\nWhat to look for:")
print("  FORMAT      — does every finding carry all 4 fields (line | issue | severity | fix),")
print("                the same way in BOTH runs? Prose spec drifts; examples pin it down.")
print("  FALSE POS   — is line 2 (parameterized, SAFE) left alone? The acceptable-pattern example")
print("                is what buys that.")
print("  GENERALIZE  — is line 7 (.format(), never demonstrated) still caught? Few-shot teaches the")
print("                PRINCIPLE, not a lookup table of shown patterns.")
print("\nNOTE (honest): on a capable model + a 3-line toy, run A often copes. The lesson is that")
print("few-shot is what makes format + judgment consistent AT SCALE — see EX3's README, which logs")
print("the same mild-wobble result for FEW_SHOT=False (aggregate 0.72-0.75 vs 0.77).")

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1: answer inconsistent output with MORE PROSE.
#   SYSTEM += "Please be consistent. Always use the exact format. Do not deviate."
#   -> the guide is explicit: when detailed instructions alone produce inconsistent output,
#      FEW-SHOT EXAMPLES are the most effective technique. Adjectives are not a format spec.

# ANTI-PATTERN 2: 20 examples, all of them EASY cases.
#   -> burns tokens and teaches nothing. 2-4 examples on the AMBIGUOUS cases (with the reasoning
#      for why this action beat the plausible alternative) is the shape that transfers.

# ANTI-PATTERN 3: expect the model to only catch the exact patterns you demonstrated.
#   -> backwards: examples teach a PRINCIPLE. Showing f-string concat also buys .format() and %.
#      (If you truly need an exhaustive list, you wanted a linter, not a model.)

# ANTI-PATTERN 4 (extraction): keep a required field required and hope prose stops the fabrication.
#   -> show an example whose source LACKS the field and whose output is null. Pair it with a
#      nullable schema (4.3). Prose alone -> the model invents a plausible value.

print("Correct: 2-4 targeted examples on the AMBIGUOUS cases, each showing the exact output "
      "format and the reasoning for the judgment — including one example of an ACCEPTABLE "
      "pattern being left alone. That buys format consistency, fewer false positives, and "
      "generalization to patterns you never showed.")

**In your own code — you already wrote this.** Exercise 3:

- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:111-133` — `FEW_SHOT_TURNS`: exactly **2**
  examples, and both are chosen for **judgment**, not decoration. The second is the ambiguous one:
  spelled-out money (`"a hundred-ish"` → `100`), **no** account id → `null` *(the empty/null skill
  in the guide's last bullet)*, and honest **low confidence** on the vague fields.
- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:135-141` — the `SYSTEM` prompt that the
  examples *demonstrate* rather than describe.
- `ccaf-prep/exercises/03-extraction-pipeline/README.md:34` — Experiment A: flip `FEW_SHOT=False`
  and watch judgment/confidence degrade (mildly, on this toy — the README says so honestly).

Open `extract.py` and note that the few-shot block is a **list of real message turns**
(`user` → `assistant` `tool_use` → `tool_result`), not a paragraph inside the system prompt.

**Self-check** (cover the answers)

1. Detailed instructions produce inconsistently formatted findings. What is the guide's *most effective* fix?
2. You show the model 2 examples of f-string SQL concatenation. Will it catch `.format()` concatenation?
3. Your reviewer keeps flagging a pattern your team considers fine. Which few-shot example fixes that?
4. Extraction keeps inventing an `account_id` that isn't in the document. Two changes — which?

<details><summary>answers</summary>

1. **Few-shot examples** — 2–4 of them, demonstrating the exact desired format (location, issue, severity, suggested fix). Adding more prose about consistency is the distractor.
2. **Yes — that's the point.** Few-shot examples make the model **generalize the principle** (untrusted input interpolated into the SQL string) rather than match only the patterns shown. If it *only* matched what you demonstrated, you'd need a linter, not a model.
3. One showing that **acceptable pattern being left alone**, with the reasoning for why it's not a finding. Examples distinguishing acceptable patterns from genuine issues are the cheapest false-positive reduction there is.
4. (a) Make the schema field **nullable** so `null` is representable (4.3), and (b) add a **few-shot example whose source document lacks the field and whose output is `null`**. A required field + prose ("don't invent values") still pressures the model to fabricate.

</details>

---
## 4.3 · Enforce structured output using tool use and JSON schemas

> **Task Statement 4.3: Enforce structured output using tool use and JSON schemas**
>
> _See the official Claude Certified Architect – Foundations exam guide for this task
> statement's full description. The plain-English unpacking below is original._


**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| `tool_use` + JSON schema = guaranteed shape | Ask for JSON in prose and you must `try/except json.loads` forever. Define a **tool** whose `input_schema` *is* your schema, and the API returns a `tool_use` block whose `.input` is **already a validated dict**. Syntax errors: gone, structurally. |
| `tool_choice: "auto"` | The model **may** call a tool — or just talk. Fine for agents; **useless as a guarantee**. |
| `tool_choice: "any"` | The model **must** call *some* tool, but picks which. This is the answer when you have **several schemas and don't know the document type** — invoice? receipt? — and you need *structure* either way. |
| `tool_choice: {"type":"tool","name":"…"}` | **Forced**: this exact tool, this exact schema. Use it when a specific extraction **must** run before an enrichment step. |
| Syntax ≠ semantics | The schema guarantees `stated_total` is a **number**. It does **not** guarantee it equals the sum of the line items, or that the vendor didn't land in the customer field. **Every semantic check is still yours** — that's 4.4's job. |
| Nullable (optional) fields | If the source may not contain a PO number, make it `["string","null"]` and leave it out of `required`. A **required** field is pressure to **fabricate**. Nullability is a hallucination control, not a nicety. |
| enum + `"other"` + detail string | Closed enums break on the first unforeseen category. `enum: [..., "other", "unclear"]` plus a free-text `detail` field gives you clean aggregation **and** an escape hatch that doesn't force a wrong bucket. |
| Format normalization **in the prompt** | The schema says `date` is a string; only the **prompt** can say *"STRICT `YYYY-MM-DD`"*. Schema constrains the type, the prompt constrains the format. You need both. |

**Run it — the same invoice through all three `tool_choice` modes (3 real calls).** The fixture is
deliberately nasty:

- the line items are **120 + 45 + 30 = 195**, but the document says **`Total due: $205.00`** → a
  **semantic** error that a perfect schema cannot catch (this is the handoff to 4.4);
- the **PO number is not in the document** ("to follow under separate cover") → the nullable field
  must come back `null`, **not** invented;
- the date is `"Mar 5, 2024"` → only the *prompt's* normalization rule turns that into ISO.

In [ ]:
# 4.3 — STRUCTURED OUTPUT via tool_use. One invoice, three tool_choice modes, three REAL calls.

INVOICE = """ACME Widgets - INVOICE #A-118
Billed to: Northwind Traders
Invoice date: Mar 5, 2024
  - Widget A x2 @ 60.00 ....... 120.00
  - Gasket set   @ 45.00 .......  45.00
  - Rush shipping @ 30.00 ......  30.00
Total due: $205.00
Notes: PO number to follow under separate cover. Paid by corporate card.
"""

# Format normalization rules live in the PROMPT (the schema can only constrain TYPES).
SYSTEM_43 = (
    "You extract invoice fields by calling a tool. Normalization rules: dates as STRICT "
    "YYYY-MM-DD; money as a number with no currency symbol; currency as a 3-letter ISO code. "
    "If a field is absent from the document, return null — never invent a value."
)

EXTRACT_INVOICE = {
    "name": "extract_invoice",
    "description": "Extract the structured fields of a supplier invoice.",
    "input_schema": {"type": "object", "properties": {
        "vendor":       {"type": "string"},
        "customer":     {"type": "string"},
        "invoice_date": {"type": "string", "description": "STRICT ISO YYYY-MM-DD."},
        "line_items":   {"type": "array", "items": {"type": "object", "properties": {
                            "description": {"type": "string"},
                            "amount": {"type": "number"}},
                         "required": ["description", "amount"]}},
        "stated_total": {"type": "number", "description": "The total as PRINTED on the document."},
        "currency":     {"type": "string", "description": "3-letter ISO code, e.g. USD."},
        # NULLABLE on purpose: the doc may not carry a PO. A *required* field is pressure to
        # FABRICATE one. Absent -> null.                                          <- 4.3 skill
        "po_number":    {"type": ["string", "null"]},
        # enum + "other"/"unclear" + a detail string = extensible categorization. <- 4.3 skill
        "expense_category": {"type": "string",
                             "enum": ["hardware", "services", "shipping", "other", "unclear"]},
        "category_detail":  {"type": ["string", "null"],
                             "description": "Free text when category is 'other' or 'unclear'."}},
        # po_number / category_detail deliberately NOT required.
        "required": ["vendor", "customer", "invoice_date", "line_items", "stated_total",
                     "currency", "expense_category"]},
}

EXTRACT_RECEIPT = {   # a SECOND schema, so tool_choice="any" has a real choice to make
    "name": "extract_receipt",
    "description": "Extract the fields of a retail receipt (merchant, purchase time, amount paid).",
    "input_schema": {"type": "object", "properties": {
        "merchant":      {"type": "string"},
        "purchased_at":  {"type": "string"},
        "amount_paid":   {"type": "number"}},
        "required": ["merchant", "purchased_at", "amount_paid"]},
}

def call(tool_choice, tools, prompt, label):
    resp = client.messages.create(model=MODEL, max_tokens=700, system=SYSTEM_43,
                                  tools=tools, tool_choice=tool_choice,
                                  messages=[{"role": "user", "content": prompt}])
    tool_blocks = [b for b in resp.content if b.type == "tool_use"]
    text = "".join(b.text for b in resp.content if b.type == "text").strip()
    print(f"\n--- {label} ---")
    print(f"   stop_reason={resp.stop_reason}   tool_use blocks={len(tool_blocks)}"
          f"   called={[b.name for b in tool_blocks] or 'NOTHING (plain text)'}")
    if text and not tool_blocks:
        print(f"   text: {text[:110]}...")
    return tool_blocks

# (1) AUTO — the model MAY call a tool, or may just talk. No guarantee of structure.
call({"type": "auto"}, [EXTRACT_INVOICE],
     "Here is a document. In one sentence, what do you make of it?\n\n" + INVOICE,
     'tool_choice="auto"  (may return TEXT -> no structural guarantee)')

# (2) ANY — the model MUST call SOME tool; it picks which. This is the answer when several
#     schemas exist and the document TYPE is unknown.                              <- 4.3 skill
any_blocks = call({"type": "any"}, [EXTRACT_INVOICE, EXTRACT_RECEIPT],
                  "Extract this document. It may be an invoice or a receipt.\n\n" + INVOICE,
                  'tool_choice="any"   (MUST call a tool; Claude chooses WHICH schema fits)')

# (3) FORCED — this exact tool, so a downstream enrichment step can rely on the shape.
forced = call({"type": "tool", "name": "extract_invoice"}, [EXTRACT_INVOICE],
              "Extract this invoice.\n\n" + INVOICE,
              'tool_choice={"type":"tool","name":"extract_invoice"}   (GUARANTEED schema)')

inv = forced[0].input          # <- already a dict. No json.loads, no try/except. That is 4.3.
print("\n" + json.dumps(inv, indent=2)[:700])

# ---- what the schema DID buy, and what it did NOT ----------------------------------------
items_sum = sum(i["amount"] for i in inv["line_items"])
print(f"\nSYNTAX  : valid by construction — .input is a dict, no parse step, no JSONDecodeError.")
print(f"NULLABLE: po_number = {inv.get('po_number')!r}  "
      f"<- absent from the doc, correctly NOT fabricated")
print(f"ENUM    : expense_category = {inv['expense_category']!r}  detail = "
      f"{inv.get('category_detail')!r}   <- 'other'/'unclear' keep the enum extensible")
print(f"NORMALIZED: invoice_date = {inv['invoice_date']!r} (prompt rule, not schema), "
      f"currency = {inv['currency']!r}")
print(f"\nSEMANTICS: line items sum to {items_sum} but stated_total = {inv['stated_total']} "
      f"-> {'MISMATCH' if abs(items_sum - inv['stated_total']) > 0.01 else 'match'}")
print("   The JSON is PERFECT and the content is WRONG. A strict schema eliminates SYNTAX errors,")
print("   never SEMANTIC ones. Catching this is 4.4's job, not the schema's.")

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1: ask for JSON in prose and parse the text.
#   resp = create(messages=[{"role":"user","content":"Return JSON: {...}"}])
#   data = json.loads(resp.content[0].text)     # markdown fences, prose preamble, trailing comma...
#   -> tool_use with an input_schema removes this ENTIRE failure class. Don't hand-roll it.

# ANTI-PATTERN 2: believe a strict schema makes the data CORRECT.
#   "The schema validated, so the invoice is right."     # FALSE
#   -> the schema proved stated_total is a NUMBER. It did not prove it equals sum(line_items),
#      nor that the vendor didn't land in the customer field. Semantics need a validator (4.4).

# ANTI-PATTERN 3: mark every field REQUIRED "so nothing is missing".
#   "required": ["vendor", "customer", "po_number", ...]
#   -> the document has no PO number, so the model INVENTS one to satisfy the schema. Nullable
#      fields exist precisely to make "not in the source" representable.

# ANTI-PATTERN 4: a closed enum with no escape hatch.
#   "enum": ["hardware", "services", "shipping"]
#   -> the first software-subscription invoice gets crammed into a wrong bucket. Add "other" +
#      a detail string (and "unclear" for genuinely ambiguous cases).

# ANTI-PATTERN 5: use tool_choice="auto" and assume you got structure.
#   -> "auto" means the model MAY answer in text. If you need a guarantee, use "any" (some tool)
#      or force the specific tool by name.

print("Correct: define the schema as a tool input_schema; force the tool when the shape must be "
      "guaranteed ('any' when the document type is unknown); make source-optional fields nullable; "
      "give enums an 'other'/'unclear' + detail escape hatch; put FORMAT rules in the prompt — and "
      "still validate the semantics yourself.")

**In your own code — you already wrote this.** Exercises 3 and 5:

- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:66-99` — `EXTRACT_TOOL`: the JSON
  `input_schema`, with `account_id` typed `["string","null"]` and **deliberately left out of
  `required`** (the comment on line 96 says exactly why: *"nullable"* → the model must not invent).
- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:194-205` — the forced call
  (`tool_choice={"type":"tool","name":"record_refund_request"}`, line 197) and the parse that
  **isn't a parse**: `next(b for b in resp.content if b.type == "tool_use")` → `.input` is already
  a dict. Line 200-202's comment is the guide bullet in prose.
- `ccaf-prep/exercises/05-cicd-review/review.py:100-123` — `REPORT_TOOL`: the same mechanism used
  for **findings** instead of extraction, with an `enum` on `severity` and `category`.
- `ccaf-prep/exercises/05-cicd-review/review.py:141-145` — the forced `report_findings` call that
  makes the CI step's `json.loads` unnecessary.

Open `extract.py:66-99` and find the three schema decisions in the guide's last two bullets:
**nullable**, **enum**, **format rules in the prompt** (that one's at `:135-141`, not in the schema).

**Self-check** (cover the answers)

1. You need structured output but the document could be an invoice *or* a receipt. Which `tool_choice`?
2. Your extraction validated against a strict schema. Can the totals still be wrong?
3. A required `account_id` keeps coming back with plausible-but-invented values. Fix?
4. What does the schema *not* control that the prompt must?

<details><summary>answers</summary>

1. `tool_choice: "any"` — the model **must** call a tool (so you always get structure) but **chooses which schema** fits the document. Forcing a specific tool would jam a receipt into the invoice schema; `"auto"` would let it answer in prose.
2. **Yes.** `tool_use` + JSON schema eliminates **syntax** errors (no `JSONDecodeError`, ever), not **semantic** ones — line items that don't sum to the total, or a value in the wrong field. You still need a validator (4.4).
3. Make the field **nullable** (`["string","null"]`) and drop it from `required`. A required field is *pressure to fabricate*: the model must put **something** there. Add a few-shot example whose source lacks the field and whose output is `null` (4.2).
4. **Format.** The schema constrains the *type* (`date` is a string); only the prompt can demand *"STRICT `YYYY-MM-DD`"*, money as a bare number, currency as a 3-letter code. Schema = type; prompt = format.

</details>

---
## 4.4 · Implement validation, retry, and feedback loops for extraction quality

> **Task Statement 4.4: Implement validation, retry, and feedback loops for extraction quality**
>
> _See the official Claude Certified Architect – Foundations exam guide for this task
> statement's full description. The plain-English unpacking below is original._


**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Retry **with error feedback** | Not "call it again and hope". Send back: the **original document**, the **failed extraction**, and the **specific validation errors** — *"`date` must be YYYY-MM-DD, got 'Mar 5 2024'"*. Then the model has something to correct **against**. |
| Semantic vs syntax errors | Syntax errors are **already gone** (4.3's `tool_use`). What's left is semantic: totals that don't add up, a vendor in the customer field. Those are what your validator is for. |
| The **limit** of retry | Retry fixes *format* and *structure* — information that **is present, wrongly shaped**. It cannot conjure information that **isn't in the document**. Looping on an absent PO number just burns money and eventually produces a **fabrication**. |
| Self-correction validation flows | Make the model **show its work in the schema**: extract `calculated_total` (sum the line items yourself) *next to* `stated_total` (as printed), plus a `conflict_detected` boolean. Now the discrepancy is a **field you can filter on**, not something you have to re-derive. |
| `detected_pattern` field | Have each finding record **which code construct triggered it** (`"bare-except"`, `"fstring-sql"`). When developers dismiss findings, you can group the dismissals **by pattern** and see *which rule* is generating your false positives — instead of guessing. |

**Run it — three flows (5 real calls).**

**(a) Self-correction schema.** Ask for `calculated_total` (the model sums the items) alongside
`stated_total` (as printed) and a `conflict_detected` boolean. The invoice from 4.3 has a planted
mismatch — watch the model **flag its own source document**.

**(b) Retry that works.** A strict validator rejects a non-ISO date; the specific error goes back
as a `tool_result` with `is_error`, and the model self-corrects. *(As in `EX3`, we corrupt attempt
0's date on purpose — `DEMO_CORRUPT` — so the retry path fires every run instead of only when the
model happens to slip. The corruption is a teaching device; the retry and the correction are real.)*

**(c) Retry that cannot work.** The PO number is **not in the document**. We make it a **required**
field and retry twice. Watch the loop fail to conjure it — and note what the model does instead.

In [ ]:
# 4.4 (a) — SELF-CORRECTION VALIDATION FLOW: calculated_total + stated_total + conflict_detected.
# Reuses INVOICE / SYSTEM_43 from 4.3 (run that cell first).

EXTRACT_V2 = json.loads(json.dumps(EXTRACT_INVOICE))          # copy 4.3's schema, then extend it
EXTRACT_V2["name"] = "extract_invoice_checked"
EXTRACT_V2["input_schema"]["properties"].update({
    "calculated_total":  {"type": "number",
                          "description": "The SUM of line_items amounts, computed by you."},
    "conflict_detected": {"type": "boolean",
                          "description": "True if calculated_total != stated_total, or the source "
                                         "data is internally inconsistent."},
})
EXTRACT_V2["input_schema"]["required"] += ["calculated_total", "conflict_detected"]

resp = client.messages.create(
    model=MODEL, max_tokens=800, system=SYSTEM_43, tools=[EXTRACT_V2],
    tool_choice={"type": "tool", "name": "extract_invoice_checked"},
    messages=[{"role": "user", "content": "Extract this invoice.\n\n" + INVOICE}])
v2 = next(b for b in resp.content if b.type == "tool_use").input

print("(a) SELF-CORRECTION FLOW — the discrepancy is now a FIELD, not a mystery:")
print(f"    stated_total     = {v2['stated_total']}   (as printed on the document)")
print(f"    calculated_total = {v2['calculated_total']}   (line items, summed by the model)")
print(f"    conflict_detected= {v2['conflict_detected']}   <- filter your pipeline on THIS")
print("    The schema (4.3) could never have caught this. Asking the model to show its arithmetic")
print("    IN the schema turns a semantic error into a queryable boolean.")

In [ ]:
# 4.4 (b) — RETRY WITH ERROR FEEDBACK: the info IS present, just mis-formatted -> retry SUCCEEDS.

DEMO_CORRUPT = True    # teaching device (mirrors EX3's DEMO_RETRY): corrupt attempt 0's date so
                       # the retry path fires every run. Flip False and attempt 0 usually passes.

def validate_invoice(d):
    """Semantic/format validation. NOT syntax — tool_use already killed that class (4.3)."""
    errors = []
    p = str(d.get("invoice_date", "")).split("-")
    if not (len(p) == 3 and len(p[0]) == 4 and p[0].isdigit()
            and len(p[1]) == 2 and p[1].isdigit() and len(p[2]) == 2 and p[2].isdigit()):
        errors.append(f"'invoice_date' must be STRICT YYYY-MM-DD, got {d.get('invoice_date')!r}.")
    if len(str(d.get("currency", ""))) != 3:
        errors.append(f"'currency' must be a 3-letter ISO code, got {d.get('currency')!r}.")
    return errors

messages = [{"role": "user", "content": "Extract this invoice.\n\n" + INVOICE}]
MAX_RETRIES = 1

for attempt in range(MAX_RETRIES + 1):
    r = client.messages.create(model=MODEL, max_tokens=700, system=SYSTEM_43,
                               tools=[EXTRACT_INVOICE],
                               tool_choice={"type": "tool", "name": "extract_invoice"},
                               messages=messages)
    tb = next(b for b in r.content if b.type == "tool_use")
    data = dict(tb.input)
    if DEMO_CORRUPT and attempt == 0:
        data["invoice_date"] = "Mar 5 2024"      # present-but-mis-formatted == RETRYABLE

    errs = validate_invoice(data)
    print(f"  [attempt {attempt}] invoice_date={data['invoice_date']!r} -> "
          f"{'VALID' if not errs else 'INVALID: ' + '; '.join(errs)}")
    if not errs:
        print(f"  -> accepted after {attempt} retr{'y' if attempt == 1 else 'ies'}")
        break

    # The follow-up request carries: the original document (already in `messages`), the FAILED
    # extraction (the assistant turn), and the SPECIFIC errors (the tool_result).   <- 4.4 skill
    messages.append({"role": "assistant", "content": r.content})
    messages.append({"role": "user", "content": [{
        "type": "tool_result", "tool_use_id": tb.id, "is_error": True,
        "content": "Validation failed. FIX and re-call: " + " ".join(errs)}]})
    print("  -> feeding the SPECIFIC errors back as a tool_result(is_error=True)")

print("\nThis retry works because the date IS in the document — just wrongly shaped. Format and")
print("structure errors are exactly what retry-with-feedback is for.")

In [ ]:
# 4.4 (c) — RETRY THAT CANNOT WORK: the info is ABSENT from the source. Two attempts, no miracle.
# The invoice says "PO number to follow under separate cover" -> the PO lives in a document we
# never provided. We make po_number REQUIRED (the anti-pattern from 4.3) and watch the loop.

EXTRACT_PO = {
    "name": "extract_po",
    "description": "Extract the purchase-order reference from an invoice.",
    "input_schema": {"type": "object", "properties": {
        "po_number": {"type": "string",
                      "description": "The PO reference, format PO-1234."}},
        "required": ["po_number"]},          # <- REQUIRED: the model MUST put something here
}

def validate_po(d):
    po = str(d.get("po_number", ""))
    ok = po.startswith("PO-") and po[3:].isdigit() and len(po[3:]) == 4
    return [] if ok else [f"'po_number' must match PO-#### and come from the document, got {po!r}."]

msgs = [{"role": "user", "content": "Extract the PO number from this invoice.\n\n" + INVOICE}]
for attempt in range(2):
    r = client.messages.create(model=MODEL, max_tokens=300, system=SYSTEM_43, tools=[EXTRACT_PO],
                               tool_choice={"type": "tool", "name": "extract_po"}, messages=msgs)
    tb = next(b for b in r.content if b.type == "tool_use")
    d = dict(tb.input)
    errs = validate_po(d)
    print(f"  [attempt {attempt}] po_number={d.get('po_number')!r} -> "
          f"{'VALID' if not errs else 'INVALID'}")
    if not errs:
        print("     (If this 'passed', LOOK AT THE VALUE: the document contains no PO number, so a")
        print("      well-formed one here is a FABRICATION — which is the failure this teaches.)")
        break
    msgs.append({"role": "assistant", "content": r.content})
    msgs.append({"role": "user", "content": [{
        "type": "tool_result", "tool_use_id": tb.id, "is_error": True,
        "content": "Validation failed. FIX and re-call: " + " ".join(errs)}]})

print("\nTHE LIMIT OF RETRY (4.4): the PO number exists only in a document we never provided.")
print("No number of retries can extract what is not there — the loop either keeps failing or")
print("pressures the model into INVENTING a plausible value. The correct design is 4.3's:")
print("  po_number: {'type': ['string','null']}, NOT in `required`  ->  null, then human review.")
print("Retry is for FORMAT/STRUCTURE errors (b). It is not a search for missing information.")

**The fourth bullet — `detected_pattern`.** The other half of a feedback loop isn't the model
correcting itself; it's **you** learning which of your rules is noisy. Add a `detected_pattern`
field to every finding (`"fstring-sql"`, `"bare-except"`, `"unclosed-file"`), log what developers
**dismiss**, and group the dismissals by pattern — now "our reviewer has too many false positives"
becomes *"the `bare-except` rule accounts for 80% of dismissals"*, which is an actionable prompt
fix (or a 4.1 category to disable). You'll see `detected_pattern` come back on real findings in
**§4.6** below.

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1: retry with the SAME prompt and no feedback.
#   for _ in range(3): data = extract(doc)      # identical request, different dice roll
#   -> the model never learns WHAT was wrong. Append the specific validation errors (and the
#      failed extraction) so the follow-up has something to correct against.

# ANTI-PATTERN 2: retry when the information is simply ABSENT.
#   while not valid(d): d = extract(doc)        # the PO number is in ANOTHER document
#   -> burns tokens, then produces a FABRICATION to satisfy the required field. Nullable + human
#      review is the answer; retry is for format/structure errors only.

# ANTI-PATTERN 3: trust the schema and skip semantic validation.
#   -> tool_use guaranteed the JSON parses. It did not check that line items sum to the total.

# ANTI-PATTERN 4: re-derive the discrepancy in Python instead of asking for it in the schema.
#   (Fine for a sum — but for "is this source internally inconsistent?" the model is the one
#    reading the document. calculated_total + conflict_detected makes its check auditable.)

# ANTI-PATTERN 5: "developers dismiss 60% of findings" with no detected_pattern field.
#   -> you cannot fix what you cannot group. Without the pattern label, every false-positive
#      post-mortem is guesswork.

print("Correct: validate semantics yourself; on failure send back the document + the failed "
      "extraction + the SPECIFIC errors; recognize when the information is absent (retry is "
      "useless -> nullable + human review); and design the schema to self-check "
      "(calculated_total vs stated_total, conflict_detected, detected_pattern).")

**In your own code — you already wrote this.** Exercise 3:

- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:163-179` — `validate()`: the **semantic /
  format** validator (strict `YYYY-MM-DD`, amount must be a number). Its docstring names the exact
  distinction the guide draws: *"These are NOT JSON syntax errors — tool_use already eliminated
  those (D4.3)."*
- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:213-228` — the retry: validation errors →
  an `assistant` turn + a `tool_result` with **`is_error: True`** carrying the specific errors →
  the model self-corrects on attempt 1.
- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:226-228` — the comment that **is** the
  guide's "limits of retry" bullet: *"this retry helps ONLY because the date/amount ARE in the
  source... If a field were ABSENT, no amount of retrying would conjure it."*
- `ccaf-prep/exercises/03-extraction-pipeline/extract.py:143-150` — `DEMO_RETRY`, the same teaching
  device used in cell (b) above, with the reasoning written out.

Run `cd ccaf-prep/exercises/03-extraction-pipeline && uv run python extract.py` and watch
`VALIDATION failed: [...] -> retrying with feedback` scroll past. That line is Task Statement 4.4.

**Self-check** (cover the answers)

1. Extraction returns a date as `"Mar 5 2024"`. Retry or redesign?
2. Extraction returns a `po_number` that isn't in the document. Retry or redesign?
3. `tool_use` guarantees valid JSON. So what is the validator still for?
4. How do you make "line items don't sum to the total" a thing your pipeline can *filter on*?
5. Developers dismiss 60% of your findings. What field should have been in the schema all along?

<details><summary>answers</summary>

1. **Retry — with feedback.** The date **is** in the source, just mis-formatted: a format error, exactly what retry-with-error-feedback fixes. Send back the document, the failed extraction, and the specific error (*"must be YYYY-MM-DD"*).
2. **Redesign.** The information is **absent** from the source (it's in a document you never provided). No retry can conjure it; looping just pressures the model into fabricating. Make the field **nullable**, let it come back `null`, and route to **human review**.
3. For **semantics**. `tool_use` eliminated the syntax class (nothing will ever fail to parse). What remains is values that don't sum, values in the wrong field, dates that parse but are impossible — none of which a schema can see.
4. Ask the **model** for `calculated_total` (its own sum of the line items) alongside `stated_total` (as printed), plus a **`conflict_detected` boolean**. The discrepancy becomes a queryable field instead of something you rediscover downstream.
5. **`detected_pattern`** — the code construct that triggered each finding. Without it you can't group dismissals by rule, so you can't tell *which* criterion is generating the noise (and 4.1's "disable the noisy category" has no target).

</details>

---
## 4.5 · Design efficient batch processing strategies

> **Task Statement 4.5: Design efficient batch processing strategies**
>
> _See the official Claude Certified Architect – Foundations exam guide for this task
> statement's full description. The plain-English unpacking below is original._


**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| 50% cost savings | Half price, for the *same* model. That's the whole temptation — and why the exam builds a distractor out of it. |
| Up to 24 hours, **no SLA** | Most batches finish in minutes. "Usually fast" is **not** a guarantee, and you cannot design a blocking gate on a hope. This single fact answers **sample Q11**. |
| Non-blocking, latency-tolerant workloads | Overnight tech-debt reports, weekly audits, nightly test generation, back-filling a year of documents. Nobody is sitting there waiting. |
| **In**appropriate for blocking workflows | A pre-merge check where a developer is staring at a spinner. Even a 20-minute batch is a broken developer experience; a 24-hour one is an outage. |
| No multi-turn tool calling in a request | The limit is **one turn, not one tool**. A batch request may include `tools` and *return* one or several `tool_use` blocks (parallel tool use is fine) — but nobody is there to **execute** them and hand the result back, so there's no *second* turn. You *could* chain batches by hand (feed the `tool_result` into a fresh batch), but each hop is ≤24h with no SLA, so an N-step loop is N days — an agentic loop (D1.1) stays on the sync API. |
| `custom_id` | Your correlation key. Results come back **unordered**; `custom_id` is how you know which output belongs to which input — and which specific documents to **resubmit** when some fail. |
| Submission frequency vs SLA | Arithmetic, and a favorite exam shape: if you promise a **30-hour** SLA and a batch may take **24**, you have **6 hours** of slack → submit every **4 hours** (not once a day) so no document ever waits longer than 6 + 24 = 30. |
| Resubmit **only** the failures | Batch results are per-request. When 3 of 500 documents fail (say, context-limit), you resubmit **those 3** — chunked — by `custom_id`. Re-running all 500 throws away the 497 you already paid for. |
| Refine the prompt on a **sample** first | Test on 10 documents synchronously, *then* batch the 10,000. A prompt bug discovered after the full run costs you the full run. |

**Run it — a REAL Message Batch (2 requests).** This is the actual `client.messages.batches` API,
not a simulation: we submit both invoices under `custom_id`s, poll for completion, and correlate
the results back by `custom_id`. Each request forces the **`extract_invoice` tool from 4.3** — so
you also see the "one turn only" constraint concretely: the batch can *return* a `tool_use` block,
but no one is there to run a tool and continue the conversation.

Most small batches land in under a minute; if this one is still `in_progress` when the poll window
expires, **that is the lesson, not a failure** — print the batch id, come back later, and note that
a pre-merge gate could never have waited. (Cost: 2 tiny requests at **half price**.)

In [ ]:
# 4.5 — a REAL Message Batch. Two documents, two custom_ids, one poll loop.

INVOICE_2 = """Globex Ltd - INVOICE #G-207
Billed to: Northwind Traders
Invoice date: April 11, 2024
  - Consulting, 3 days @ 400.00 ... 1200.00
  - Travel .......................... 250.00
Total due: $1450.00
Notes: PO-4417 authorized by finance.
"""

DOCS = {"invoice-A-118": INVOICE, "invoice-G-207": INVOICE_2}     # custom_id -> document

# Each request is ONE turn: system + tools + forced tool_choice, exactly like the sync call in 4.3.
batch = client.messages.batches.create(requests=[
    {"custom_id": cid,                                            # <- the correlation key
     "params": {"model": MODEL, "max_tokens": 700, "system": SYSTEM_43,
                "tools": [EXTRACT_INVOICE],
                "tool_choice": {"type": "tool", "name": "extract_invoice"},
                "messages": [{"role": "user", "content": "Extract this invoice.\n\n" + doc}]}}
    for cid, doc in DOCS.items()])

print(f"submitted batch {batch.id}   status={batch.processing_status}")
print(f"  requests: {batch.request_counts}")
print("  billed at 50% of the sync price; the window is up to 24h with NO latency SLA.\n")

POLL_SECONDS = int(os.environ.get("BATCH_POLL_SECONDS") or 180)   # bounded so the notebook ends
waited = 0
while waited < POLL_SECONDS:
    batch = client.messages.batches.retrieve(batch.id)
    if batch.processing_status == "ended":
        break
    print(f"  ...{batch.processing_status} after {waited}s  (this wait is exactly why a BLOCKING "
          f"pre-merge check cannot use batch)")
    time.sleep(15); waited += 15

if batch.processing_status == "ended":
    print(f"\nended after ~{waited}s. counts={batch.request_counts}")
    print("results, correlated by custom_id (they arrive UNORDERED — the id is how you match):\n")
    failed = []
    for r in client.messages.batches.results(batch.id):
        if r.result.type == "succeeded":
            blocks = r.result.message.content
            tb = next((b for b in blocks if b.type == "tool_use"), None)
            inv = tb.input if tb else {}
            print(f"  {r.custom_id:<15} OK   vendor={inv.get('vendor')!r} "
                  f"total={inv.get('stated_total')} po={inv.get('po_number')!r}")
            print(f"  {'':<15}      stop_reason={r.result.message.stop_reason!r} <- the turn ENDS on"
                  " the tool_use block; nothing can execute the tool and continue (no multi-turn)")
        else:
            failed.append(r.custom_id)
            print(f"  {r.custom_id:<15} {r.result.type.upper()}")
    print(f"\n  resubmit list (by custom_id): {failed or 'none — nothing to resubmit'}")
    print("  ^ when 3 of 500 documents fail, you resubmit THOSE 3 (chunked, if they blew the")
    print("    context limit) — not all 500. That is what custom_id buys you.")
else:
    print(f"\nstill {batch.processing_status} after {POLL_SECONDS}s — and THAT is task statement 4.5.")
    print("There is NO latency SLA (up to 24h). A nightly report does not care; a developer")
    print("waiting on a pre-merge gate does. Retrieve it later with:")
    print(f'    client.messages.batches.results("{batch.id}")')

**The routing rule (this is sample Q11, in one table).** Two workflows, two answers — the mistake
the question punishes is applying **one** answer to both:

| Workflow | Blocking? | API | Why |
|---|---|---|---|
| Pre-merge review gate | **Yes** — a developer is waiting | **Synchronous** | No SLA on batch. "Usually a few minutes" is not something you can gate a merge on. Pay full price; it's the cost of interactivity. |
| Overnight tech-debt report | No — read tomorrow morning | **Batch** (50% off) | 24 hours of slack, nobody waiting. This is exactly the workload the discount exists for. |
| Multi-turn agentic loop (D1.1) | either | **Synchronous** | A batch request is one turn — no one can execute a tool mid-request and hand back the result. |

**The SLA arithmetic (why "submit every 4 hours", spelled out).** The trap in this shape is
measuring from the wrong clock. The SLA is a promise to each **document** — *"from the moment you
arrive, you get an answer within 30 hours"* — **not** a promise about any one batch. So a document's
total wait has two parts:

> `document wait = (time until the next batch departs) + (up to 24h processing)`

The second part is fixed by the Batch API. The **first** part is the one *you* control with your
submission frequency, and the worst case is a document that arrives **just after** a batch left — it
missed that bus and must wait a full interval for the next one.

| Submit every… | Worst wait for the next batch | + processing | Total | ≤ 30h? |
|---|---|---|---|---|
| **24h** (once a day) | ~24h | 24h | **~48h** | ❌ |
| **4h** | ~4h | 24h | **28h** | ✅ |

So the recipe is: slack = `30 − 24 = 6h` (the most a document can sit *before* its batch starts);
the submission interval must be **≤ that slack**. At exactly 6h you'd hit `6 + 24 = 30`, right on the
line; **4h** leaves a 2-hour buffer → `4 + 24 = 28`.

**And yes — the batches overlap, which is fine.** The 4-hour batch is still running when the 8-hour
one launches; they're independent async jobs and the provider runs many at once. Overlap isn't the
thing you're solving for — **queue wait** is. Submitting more often doesn't make any batch faster; it
just means a freshly-arrived document never has to wait long to *board* one. The exam gives you the
SLA and the batch duration, you subtract to get the slack, and the slack is the **ceiling on your
interval** — answer smaller than it, never equal or larger.

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1 (sample Q11, option B): move the BLOCKING pre-merge check to batch for the 50%.
#   -> up to 24h, NO SLA. "Batches are usually fast" is not a guarantee you can gate a merge on.
#      Poll all you like: you cannot poll your way to a latency guarantee that doesn't exist.

# ANTI-PATTERN 2 (option D): batch it, with a "timeout fallback to real-time if it's slow".
#   -> you now maintain two paths, pay for both on every slow batch, and the p99 developer
#      experience is still awful. Match each workflow to its API instead. Simpler is correct.

# ANTI-PATTERN 3 (option C): keep BOTH on sync "because batch results come back out of order".
#   -> a misconception: custom_id correlates request to response. Ordering is a solved problem.

# ANTI-PATTERN 4: run an agentic tool LOOP inside a batch request.
#   -> a batch request is ONE turn. It can return a tool_use block, but nothing executes the tool
#      and continues. Multi-turn tool calling stays on the sync API.

# ANTI-PATTERN 5: 3 of 500 documents failed -> resubmit all 500.
#   -> resubmit the 3, by custom_id, with a fix (chunk the ones that exceeded the context limit).

# ANTI-PATTERN 6: batch 10,000 documents on an unrefined prompt.
#   -> refine on a sample of ~10 SYNCHRONOUSLY first. A prompt bug found after the big run means
#      paying for the big run twice — which is a great way to spend the 50% you just saved.

print("Correct: match the API to the LATENCY requirement — sync for blocking (pre-merge) work, "
      "batch for latency-tolerant work (overnight/weekly). Correlate with custom_id, resubmit only "
      "the failures, and refine the prompt on a sample before you batch the volume.")

**In your own code — you already wrote this.** Exercise 5 is the *sync* half of the rule, and its
README argues the *batch* half explicitly:

- `ccaf-prep/exercises/05-cicd-review/ci_review.sh` — the pre-merge gate. It runs **headless and
  synchronous** (`claude -p`, `--output-format json`) and blocks the merge on its exit code. This is
  the workflow the guide says must **not** be moved to batch.
- `ccaf-prep/exercises/05-cicd-review/README.md:98-101` — Pitfall 4, *"Batch the blocking gate
  (Q11, D4.5)"*: the temptation, and why it's wrong (async, ≤24h, no SLA, no multi-turn tools).
- `ccaf-prep/exercises/05-cicd-review/README.md:108-110` — *"when batch IS right"*: offline/nightly
  report generation across many files, where a 24h window is free.
- `ccaf-prep/exercises/03-extraction-pipeline/extract.py` — the natural batch citizen: one document
  in, one structured extraction out, nobody waiting. Exactly the shape the cell above submitted.

**Self-check** (cover the answers)

1. Your manager wants both the pre-merge check *and* the overnight report on the Batches API for the 50% discount. Your call?
2. "Batches usually finish in minutes, so let's poll and use it for the merge gate." What's wrong?
3. You promise a 30-hour turnaround; a batch may take 24. How often do you submit?
4. 3 of 500 documents failed with a context-limit error. What do you resubmit?
5. Why can't your D1.1 agentic loop run inside a batch request?

<details><summary>answers</summary>

1. **Batch the report, keep the pre-merge check synchronous.** The discount is real, but the Batches API has **no latency SLA** (up to 24h) — fine for a report read tomorrow morning, unacceptable for a gate a developer is waiting on. (Sample Q11, answer A.)
2. *"Usually"* is not an SLA. A blocking workflow needs a **guarantee**, and polling doesn't create one — it just makes the wait observable. The API's contract is "up to 24 hours".
3. Every **4 hours**. 30-hour SLA − 24-hour worst-case processing = **6 hours** of slack for a document to sit in a pending batch; a 4-hour submission window fits inside it with room to spare.
4. **Those 3, by `custom_id`**, with a fix — chunk the documents that exceeded the context limit. Resubmitting all 500 throws away 497 completed results you already paid for.
5. A batch request is **one turn**. It can *return* a `tool_use` block, but there's no live loop to **execute** the tool and feed the `tool_result` back — so multi-turn tool calling can't happen inside it. Agentic loops stay on the synchronous API.

</details>

---
## 4.6 · Design multi-instance and multi-pass review architectures

> **Task Statement 4.6: Design multi-instance and multi-pass review architectures**
>
> _See the official Claude Certified Architect – Foundations exam guide for this task
> statement's full description. The plain-English unpacking below is original._


**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Self-review limitation | *"Now review the code you just wrote"* runs **in the same session**, carrying the reasoning that produced the bug. The model already "decided" the loop bound was right — asking it to re-check tends to **re-justify** rather than re-derive. |
| Independent instance | A **fresh conversation** that sees the **code only** — no author's rationale, no "as we discussed". It has nothing to defend, so it reads what's actually there. Cheap, and strictly better than *"please double-check yourself"*. |
| More effective than extended thinking | Note what the guide rules out: more thinking **inside the same context** doesn't fix it either. The problem is the **contaminated context**, not the amount of reasoning. |
| Multi-pass: per-file + integration | 14 files in one prompt → **attention dilutes** (detailed on file 1, superficial on file 9) and findings **contradict** (a pattern flagged in one file, approved in another). Fix: **one call per file** for local issues, then a **separate** pass over all files for cross-file data flow. |
| The integration pass is not a merge | It's a **new question** the per-file passes structurally could not answer: *does the value one file returns match the shape another file consumes?* You collect N local results **plus** 1 cross-cutting result. |
| Self-reported confidence **for routing** | Confidence is a bad **filter** (4.1: poorly calibrated) but a useful **router**: high-confidence findings post as PR comments automatically, low-confidence ones go to a human triage queue. Nothing is silently dropped. |

**Run it (1/2) — self-review vs an independent instance (2 real calls).** A clean A/B with **one**
variable.

Both reviewers get the **identical system prompt** (4.1's criteria) and the **identical code**. The
only difference is the **message list**:

- the **self-reviewer**'s list contains the authoring turn — the code *and the rationale that
  produced the bug* (*"I subtracted 1 from `end` so the slice stops at the last index of the
  page"*). It is being asked to contradict itself.
- the **independent instance** sees the code and the requirement. Nothing to defend.

*Teaching simplification (as in `EX5`): the authored code and its rationale are a **fixture** with a
**planted** off-by-one — otherwise a clean generation would leave the A/B with nothing to detect.
The fixture is the prior conversation; **both reviews are real calls**, and what each one reports is
Claude's decision, not ours.*

In [ ]:
# 4.6 (1) — SELF-REVIEW vs INDEPENDENT INSTANCE. ONE variable: is the author's reasoning in the
# message list? Same code, same criteria, same schema, same model. Both reviews are REAL calls.

SPEC = ("Write `page(items, page_num, per_page)`: return the items on page `page_num` "
        "(1-indexed), and append one line to /tmp/page.log per call.")

# FIXTURE (planted, as in EX5): the "authored" code, with an off-by-one that drops the last item
# of every page, and a file handle that is never closed.
AUTHORED_CODE = """def page(items, page_num, per_page):
    log = open("/tmp/page.log", "a")
    log.write(f"page {page_num}\\n")
    start = (page_num - 1) * per_page
    end = start + per_page - 1
    return items[start:end]
"""

# FIXTURE: the authoring turn — the code PLUS the reasoning that produced the bug. THIS is the
# contamination the guide describes: the model already argued itself into the wrong slice bound.
AUTHOR_TURN = ("```python\n" + AUTHORED_CODE + "```\n"
               "I subtracted 1 from `end` so the slice stops at the last index of the page rather "
               "than running past it, and I open the log in append mode so each call adds a line.")

# The findings schema for both reviewers. Two 4.4 fields ride along: detected_pattern (which
# construct triggered it) and confidence (for the routing demo in the next cell).
VERIFY_TOOL = {
    "name": "report_findings",
    "description": "Report code-review findings.",
    "input_schema": {"type": "object", "properties": {"findings": {"type": "array", "items": {
        "type": "object", "properties": {
            "issue":            {"type": "string"},
            "category":         {"type": "string", "enum": list(CRITERIA)},
            "detected_pattern": {"type": "string",
                                 "description": "The code construct that triggered this, e.g. "
                                                "'off-by-one-range' or 'unclosed-file'."},
            "confidence":       {"type": "number", "description": "0.0-1.0, your calibrated "
                                                                  "confidence this is a real bug."}},
        "required": ["issue", "category", "detected_pattern", "confidence"]}}},
        "required": ["findings"]},
}

# IDENTICAL system prompt for both reviewers (4.1's criteria + the requirement under review).
REVIEW_SYS = (
    "You are a code reviewer. Report a finding ONLY if it falls in one of these categories:\n"
    + "\n".join(f"  - {k}: {v}" for k, v in CRITERIA.items())
    + "\n\nThe requirement under review: `page(items, page_num, per_page)` must return ALL "
      "`per_page` items of the 1-indexed page `page_num`, and append one line to /tmp/page.log."
)

def review_with(messages, label):
    r = client.messages.create(model=MODEL, max_tokens=600, system=REVIEW_SYS, tools=[VERIFY_TOOL],
                               tool_choice={"type": "tool", "name": "report_findings"},
                               messages=messages)
    tb = next((b for b in r.content if b.type == "tool_use"), None)
    fs = tb.input.get("findings", []) if tb else []
    print(f"\n--- {label}: {len(fs)} finding(s) ---")
    for f in fs:
        print(f"   [{f['confidence']:.2f}] {f['category']:<13} {f['detected_pattern']:<22} "
              f"{f['issue'][:54]}")
    return fs

# (A) SELF-REVIEW — the author's own session: its code AND its rationale are in the context.
self_findings = review_with(
    [{"role": "user", "content": SPEC},
     {"role": "assistant", "content": AUTHOR_TURN},                  # <- the contamination
     {"role": "user", "content": "Now review the code you just wrote and report any findings."}],
    "SELF-REVIEW (same session — its own code + rationale in context)")

# (B) INDEPENDENT INSTANCE — fresh message list. Same code, no author, no rationale.
indep_findings = review_with(
    [{"role": "user", "content": "Review this code and report any findings.\n\n```python\n"
                                 + AUTHORED_CODE + "```"}],
    "INDEPENDENT INSTANCE (fresh context — the CODE only)")

# The planted off-by-one is the only 'correctness' bug in the file (the other is 'resource-leak'),
# so Claude's OWN category label tells us whether each reviewer caught it.
caught = lambda fs: any(f["category"] == "correctness" for f in fs)
print(f"\nplanted off-by-one (`end = start + per_page - 1` drops the last item of every page):")
print(f"   SELF-REVIEW  caught it? {caught(self_findings)}")
print(f"   INDEPENDENT  caught it? {caught(indep_findings)}")
print("\nMECHANISM (real, deterministic): the ONLY difference between those two calls is whether")
print("  the author's reasoning sits in the message list. Same model, same criteria, same code.")
print("  A model that has just ARGUED FOR the wrong slice bound tends to re-justify it rather than")
print("  re-derive it — the guide's 'less likely to question its own decisions'.")
print("OUTCOME (model-dependent): on a 6-line toy the self-reviewer sometimes catches it anyway.")
print("  The architectural rule does not depend on this run: never let the author be the only")
print("  reviewer, and don't reach for 'think harder' INSIDE the contaminated session instead.")

**Run it (2/2) — per-file passes + a separate integration pass (3 real calls).** This is **sample
Q12**. Two files: each has a **local** bug, and *together* they have a **contract mismatch**
(`charge()` returns a bare `bool`; `settle()` dereferences `result["ok"]`). No single-file pass can
see the mismatch — it isn't *in* either file, it's *between* them.

Then we route the findings by the model's **self-reported confidence**: high → auto-post as a PR
comment; low → human triage queue. (Confidence is a bad *filter* — 4.1 — but a fine *router*.)

In [ ]:
# 4.6 (2) — MULTI-PASS: N per-file LOCAL passes + 1 cross-file INTEGRATION pass. (Sample Q12.)

PAYMENTS = """# payments.py
1  def charge(account, amount):
2      log = open("/tmp/charge.log", "a")        # never closed
3      log.write(f"charging {amount}\\n")
4      account["balance"] -= amount
5      return True                               # returns a bare bool
"""
LEDGER = """# ledger.py
1  from payments import charge
2
3  def settle(account, amount):
4      result = charge(account, amount - 1)      # off-by-one
5      if result["ok"]:                          # expects a dict with "ok"
6          return result["txn_id"]
7      return None
"""
FILES = {"payments.py": PAYMENTS, "ledger.py": LEDGER}

def pass_(system, user, cats):
    tool = json.loads(json.dumps(VERIFY_TOOL))
    tool["input_schema"]["properties"]["findings"]["items"]["properties"]["category"]["enum"] = cats
    r = client.messages.create(model=MODEL, max_tokens=600, system=system, tools=[tool],
                               tool_choice={"type": "tool", "name": "report_findings"},
                               messages=[{"role": "user", "content": user}])
    tb = next((b for b in r.content if b.type == "tool_use"), None)
    return tb.input.get("findings", []) if tb else []

INDEP = ("You are an INDEPENDENT code reviewer. Report findings ONLY in these categories:\n"
         + "\n".join(f"  - {k}: {v}" for k, v in CRITERIA.items()))

print("-- PER-FILE LOCAL PASSES (one call per file: attention cannot dilute) --")
local = []
for name, src in FILES.items():
    fs = pass_(INDEP, f"Review THIS ONE FILE ({name}) for LOCAL issues only.\n\n{src}",
               list(CRITERIA))
    print(f"   [{name}] {len(fs)} local finding(s)")
    for f in fs:
        print(f"      [{f['confidence']:.2f}] {f['category']:<13} {f['detected_pattern']:<20} "
              f"{f['issue'][:52]}")
    local += fs

print("\n-- CROSS-FILE INTEGRATION PASS (a NEW question, not a merge of the above) --")
integ = pass_(INDEP + "\n  - cross-file: a value returned by one file is consumed with a different "
                      "shape/type by another.",
              "These files ship together. Find ONLY CROSS-FILE contract mismatches: a value one "
              "file returns but another consumes with a different shape or type.\n\n"
              + "\n".join(f"=== {n} ===\n{s}" for n, s in FILES.items()),
              list(CRITERIA) + ["cross-file"])
for f in integ:
    print(f"      [{f['confidence']:.2f}] {f['category']:<13} {f['detected_pattern']:<20} "
          f"{f['issue'][:52]}")

caught = any(f["category"] == "cross-file" for f in integ)
print(f"\ncross-file contract bug caught by the integration pass? {caught}")
print("   It is NOT in payments.py and NOT in ledger.py — it is BETWEEN them. A per-file pass")
print("   structurally cannot see it, and one big all-files prompt dilutes attention until it")
print("   misses it (that is Q12's option C, 'just use a bigger context window').")

# ---- confidence as a ROUTER (not a filter) -------------------------------------------------
print("\n-- CALIBRATED ROUTING on the model's self-reported confidence --")
ROUTE_AT = 0.80
auto  = [f for f in local + integ if f["confidence"] >= ROUTE_AT]
triage= [f for f in local + integ if f["confidence"] <  ROUTE_AT]
print(f"   AUTO-POST as PR comments (>= {ROUTE_AT}): {len(auto)}")
print(f"   HUMAN TRIAGE queue      (<  {ROUTE_AT}): {len(triage)}")
print("   Confidence ROUTES here — nothing is silently dropped. Using it to FILTER (discard the")
print("   low-confidence ones) is 4.1's anti-pattern: LLM confidence is poorly calibrated.")
print(f"\n   detected_pattern histogram (4.4): "
      f"{sorted({f['detected_pattern'] for f in local + integ})}")

**The anti-patterns (exam distractors):**

In [ ]:
# ANTI-PATTERN 1 (sample Q12, option C): "switch to a bigger model / bigger context window".
#   -> a larger window fits all 14 files; it does not give each file ATTENTION. Inconsistent depth
#      and contradictory findings persist. The fix is SPLITTING the review, not scaling the input.

# ANTI-PATTERN 2 (sample Q12, option D): run 3 full passes, keep only findings that appear in 2+.
#   -> consensus voting SUPPRESSES real bugs: a subtle issue caught intermittently (once out of
#      three) is exactly the kind you most need — and this rule throws it away.

# ANTI-PATTERN 3 (sample Q12, option B): make developers split their PRs into 3-4 files.
#   -> pushes your system's problem onto humans. The review architecture is what's broken.

# ANTI-PATTERN 4: "review your own code carefully" in the same session.
#   messages = [spec, assistant_code, {"role":"user","content":"now double-check your work"}]
#   -> it carries the reasoning that produced the bug. Self-review re-justifies. Use a SECOND,
#      INDEPENDENT instance that sees the code only.

# ANTI-PATTERN 5: reach for extended thinking instead of an independent instance.
#   -> more reasoning inside the SAME contaminated context. The guide is explicit that an
#      independent instance beats both self-review instructions and extended thinking here.

# ANTI-PATTERN 6: DISCARD low-confidence findings.
#   findings = [f for f in findings if f["confidence"] > 0.8]
#   -> poorly calibrated (4.1). ROUTE them to human triage; don't delete them.

print("Correct: review with a SECOND, INDEPENDENT instance that never saw the author's reasoning; "
      "split large reviews into per-file local passes PLUS a separate cross-file integration pass; "
      "and use self-reported confidence to ROUTE findings (auto-post vs human triage), not to "
      "filter them away.")

**In your own code — you already wrote this.** Exercise 5 *is* this task statement:

- `ccaf-prep/exercises/05-cicd-review/review.py:132-149` — `_review()`: every review runs in a
  **fresh message list**, with a system prompt that says *"You are an INDEPENDENT code reviewer. You
  did not write this code and have no access to the author's reasoning."* That's the 4.6 skill in
  one function.
- `ccaf-prep/exercises/05-cicd-review/review.py:152-182` — `review_decomposed()`: the per-file local
  passes (each file in its **own call**, `allow_cross_file=False`) **plus** the separate integration
  pass that only looks for contract mismatches.
- `ccaf-prep/exercises/05-cicd-review/review.py:185-199` — `review_monolith()`: the **anti-pattern**,
  all files in one big prompt. Flip `DECOMPOSE = False` (line 56) and run it — the README logs
  **3/3 runs returning zero findings** on Haiku, the cross-file bug diluted away.
- `notebooks/D1_agentic_loops.ipynb` §1.6 — the same split seen from the **orchestration** side
  (fixed prompt-chaining). 4.6 and 1.6 are two views of one idea, which is why **sample Q12 counts
  for both domains**.

Run it: `cd ccaf-prep/exercises/05-cicd-review && uv run python review.py`, then flip `DECOMPOSE`.

**Self-check** (cover the answers)

1. A 14-file PR gets detailed feedback on some files, superficial on others, and contradictory findings. Fix?
2. Why doesn't a bigger context window fix that?
3. Why is *"now review your own code carefully"* weak — and why doesn't extended thinking rescue it?
4. Why not run three full reviews and keep only the findings that appear in at least two?
5. Your reviewer self-reports confidence per finding. What do you do with it?

<details><summary>answers</summary>

1. **Split into focused passes**: one **per-file local** pass per file (consistent depth, no dilution), then a **separate integration pass** for cross-file data flow. (Sample Q12, answer A.)
2. Because the problem is **attention**, not capacity. All 14 files *fit* already; what degrades is how much attention each one gets. A larger window changes the first, not the second.
3. Self-review happens **in the same session**, carrying the reasoning that produced the bug — the model re-justifies rather than re-derives. Extended thinking doesn't help because it adds reasoning **inside the same contaminated context**; the fix is a **fresh instance** that sees only the code.
4. **Consensus voting suppresses real bugs.** A subtle issue caught in only one of three runs is precisely the one you need, and a "2-of-3" rule deletes it. (Sample Q12, option D.)
5. **Route with it, don't filter with it.** High confidence → auto-post as a PR comment; low → human triage. Discarding low-confidence findings relies on LLM confidence being calibrated, which 4.1 tells you it isn't.

</details>

---
## ✅ Domain 4 complete — and with it, all five domains

You made every 4.x task statement observable:

- **4.1** — explicit **report/skip criteria** + a severity rubric with concrete examples beat
  *"be conservative"*; a high-false-positive category gets **disabled**, not softened.
- **4.2** — **2 few-shot examples** on the *ambiguous* cases pin the output format, kill the
  acceptable-pattern false positive, and **generalize** to patterns you never showed.
- **4.3** — `tool_use` + `input_schema` makes malformed JSON **unrepresentable**; `auto` / `any` /
  forced are three different guarantees; nullable fields and `other`/`unclear` enums stop
  fabrication — and the schema still cannot make the numbers **true**.
- **4.4** — validate the **semantics**, retry **with the specific errors**, and know the limit:
  retry fixes *mis-shaped* information, never *absent* information.
- **4.5** — a **real Message Batch**: 50% off, no SLA, `custom_id` correlation, one turn only.
  Sync for blocking work, batch for the overnight report.
- **4.6** — an **independent instance** beats self-review; **per-file + integration passes** beat one
  big prompt; confidence **routes**, it doesn't filter.

**Threads that continue elsewhere:**
- **D1 §1.6** — the decomposition behind 4.6's multi-pass review, from the orchestration side.
  (**Sample Q12 counts for both** — that's not a coincidence, it's the same principle.)
- **D5 §5.5** — the confidence routing in 4.6 is D5's calibration story; `EX3` runs both at once.
- **D5 §5.2** — 4.1's explicit criteria are what make escalation calibrate (**sample Q3**, also
  counted in both domains).
- **D2 §2.3** — `tool_choice` `auto`/`any`/forced first appears there as a *tool-distribution*
  control; here it's a *structured-output* guarantee. Same flag, two lenses.

**Next:** the last exercise wave — **Ex3** (D4+D5) and **Ex5** (D3+D4) can now both be run.
Then `mock_exam_and_review.ipynb` and `practice_exam_A.ipynb`.

## Sample exam questions — Domain 4
<!-- ccaf:redacted -->

See the official **Claude Certified Architect – Foundations** exam guide for its sample
questions. For hands-on practice on this domain, use **`practice_exam_A.ipynb`** — original
scenario-based items.

## Exercises that use this domain

Exercises are **cross-domain** — none is D4-only — so do them once all their domains are studied
(see [`README.md`](./README.md) for the wave order).

| Exercise | Domains it reinforces | Do it after you've studied |
|----------|-----------------------|-----------------------------|
| **Ex3** `../exercises/03-extraction-pipeline/` | D4 + D5      | D5, **D4** |
| **Ex5** `../exercises/05-cicd-review/`         | D3 + D4      | D3, **D4** |

D4 is the **last** domain in the study order (`D1 → D2 → D5 → D3 → D4`), so finishing it unlocks the
final wave: **Ex3** and **Ex5** can be done **in parallel** — nothing else is blocking. Ex3 is 4.2/4.3/4.4
end to end (few-shot → forced tool → validation-retry → confidence routing); Ex5 is 4.1/4.5/4.6
(explicit criteria → per-file + integration passes → sync-vs-batch).